In [3]:
import pandas as pd

In [4]:
spy=pd.read_csv("data/SPY_data2.csv")
cpi=pd.read_csv("data/release_dates_15.csv")
spy=spy.rename(columns={ "DlyCalDt": "date",
    "DlyRet": "return"})
print(spy.head())
print(cpi.head())

   PERMNO  HdrCUSIP Ticker  PERMCO        date    return
0   84398  78462F10    SPY   46699  2010-01-04  0.016960
1   84398  78462F10    SPY   46699  2010-01-05  0.002647
2   84398  78462F10    SPY   46699  2010-01-06  0.000704
3   84398  78462F10    SPY   46699  2010-01-07  0.004221
4   84398  78462F10    SPY   46699  2010-01-08  0.003328
  release_dates
0    2010-01-15
1    2010-02-17
2    2010-02-19
3    2010-03-18
4    2010-04-14


In [6]:
spy["date"] = pd.to_datetime(spy["date"])
cpi["release_dates"] = pd.to_datetime(cpi["release_dates"])
print(spy.dtypes)
print(cpi.dtypes)

PERMNO               int64
HdrCUSIP            object
Ticker              object
PERMCO               int64
date        datetime64[ns]
return             float64
dtype: object
release_dates    datetime64[ns]
dtype: object


In [7]:
spy["return"]=pd.to_numeric(spy["return"], errors='coerce')
print ("missing values:", spy["return"].isnull().sum())

missing values: 0


In [9]:
print("Duplicate SPY dates:", spy["date"].duplicated().sum())
print("Duplicate CPI dates:", cpi["release_dates"].duplicated().sum())

Duplicate SPY dates: 0
Duplicate CPI dates: 0


In [11]:
data = spy.merge(
    cpi,
    left_on="date",
    right_on="release_dates",
    how="left",
    validate="one_to_one",
    indicator=True
)
data.to_csv("data/analysis_data_long.csv", index=False)

In [12]:

data["cpi_day"]=(data["_merge"]=="both").astype(int)
print(data["cpi_day"].value_counts())

cpi_day
0    3822
1     202
Name: count, dtype: int64


In [13]:
group_stats = data.groupby("cpi_day")["return"].agg(
    ["count", "mean", "std"]
)

print(group_stats)

         count      mean       std
cpi_day                           
0         3822  0.000587  0.010821
1          202  0.000472  0.011258


Now we will compute the regression on returns: 


Return_t =alpha+ beta* CpiDay_t +epsilon_t

In [14]:
import statsmodels.api as sm


In [15]:
reg_data = data[["return", "cpi_day"]].dropna().copy()
y = reg_data["return"]

X = reg_data[["cpi_day"]]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit(cov_type="HC1")

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 return   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                   0.02014
Date:                Fri, 02 Oct 2026   Prob (F-statistic):              0.887
Time:                        16:11:00   Log-Likelihood:                 12497.
No. Observations:                4024   AIC:                        -2.499e+04
Df Residuals:                    4022   BIC:                        -2.498e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      3.351      0.0

In [16]:
ordinary_mean = reg_data.loc[
    reg_data["cpi_day"] == 0, "return"
].mean()

cpi_mean = reg_data.loc[
    reg_data["cpi_day"] == 1, "return"
].mean()

difference = cpi_mean - ordinary_mean

print("Ordinary-day mean:", ordinary_mean * 100, "%")
print("CPI-day mean:", cpi_mean * 100, "%")
print("Difference:", difference * 100, "percentage points")
print("Regression coefficient:", model.params["cpi_day"])

Ordinary-day mean: 0.058657064364207215 %
CPI-day mean: 0.047170297029702976 %
Difference: -0.011486767334504237 percentage points
Regression coefficient: -0.00011486767334504191


In [17]:
beta = model.params["cpi_day"]
two_sided_p = model.pvalues["cpi_day"]

if beta > 0:
    one_sided_p = two_sided_p / 2
else:
    one_sided_p = 1 - two_sided_p / 2

print("CPI-day coefficient:", beta)
print("Two-sided p-value:", two_sided_p)
print("One-sided p-value:", one_sided_p)

CPI-day coefficient: -0.00011486767334504191
Two-sided p-value: 0.8871599053515506
One-sided p-value: 0.5564200473242247


on a larger sample the CPI return are even less than ordinary days and the p values also show it as statistically insignificant 